# P91-Tier-1: Game-Theoretic Contract Formulation

## Problem Description

The Contract Design to Mitigate the Bullwhip Effect addresses how supply chain partners can design contractual agreements that reduce demand variability amplification across the supply chain. The bullwhip effect occurs when orders to suppliers tend to vary more than sales to buyers, causing inefficiencies in inventory management, production planning, and distribution.

In Tier 1, we use **game-theoretic contract formulation** to model the strategic interactions between supply chain partners and design contracts that align incentives while reducing the bullwhip effect.

## Key Assumptions

1. **Supply Chain Structure**: Multi-tier supply chain with manufacturer, distributor, retailer, and supplier
2. **Demand Pattern**: Stochastic customer demand with known distribution
3. **Information Sharing**: Partners share relevant information through contract mechanisms
4. **Rational Behavior**: All parties act to maximize their own utility
5. **Contract Parameters**: Revenue sharing, flexibility provisions, and information sharing incentives

## Approach

### Game-Theoretic Framework

We model the supply chain as a **non-cooperative game** where:
- **Players**: Supply chain partners (manufacturer, distributor, retailer, supplier)
- **Strategies**: Contract design choices (revenue sharing, flexibility, information sharing)
- **Payoffs**: Profit minus costs including bullwhip-related inefficiencies
- **Equilibrium**: Nash equilibrium where no player can unilaterally improve their outcome

### Contract Design Variables

1. **Revenue Sharing Ratio (α)**: Proportion of revenue shared between partners
2. **Flexibility Factor (β)**: Degree of order quantity flexibility allowed
3. **Safety Stock Multiplier (γ)**: Buffer stock level for demand uncertainty
4. **Information Sharing Bonus (δ)**: Incentive for sharing demand information
5. **Lead Time Buffer (λ)**: Extra time for order processing and delivery

### Mathematical Formulation

#### Player Utility Functions

For each player *i* in the supply chain:

```
U_i = R_i - C_i - BW_i
```

Where:
- *R_i* = Revenue for player *i*
- *C_i* = Operating costs for player *i*
- *BW_i* = Bullwhip-related costs for player *i*

#### Bullwhip Effect Measurement

```
BW = σ_orders / σ_sales
```

Where:
- *σ_orders* = Standard deviation of orders
- *σ_sales* = Standard deviation of sales

#### Nash Equilibrium Conditions

For each player *i*:

```
U_i(s_i*, s_-i) ≥ U_i(s_i, s_-i) for all s_i
```

Where:
- *s_i** = Optimal strategy for player *i*
- *s_-i* = Strategies of all other players

## Expected Results

1. **Optimal Contract Parameters**: Values for α, β, γ, δ, λ that minimize bullwhip effect
2. **Equilibrium Analysis**: Nash equilibrium strategies for all supply chain partners
3. **Bullwhip Reduction**: Quantified reduction in demand variability amplification
4. **Profit Improvement**: Increased profitability for all supply chain partners
5. **Sensitivity Analysis**: Impact of parameter changes on equilibrium outcomes

## Concrete Example

### Scenario: Electronics Supply Chain

**Supply Chain Structure:**
- **Supplier**: Component manufacturer
- **Manufacturer**: Device assembly
- **Distributor**: Regional distribution
- **Retailer**: Consumer sales

**Demand Characteristics:**
- **Base Demand**: 1000 units/period
- **Demand Variability**: σ = 200 units
- **Lead Time**: 2 periods
- **Review Period**: 1 period

**Contract Parameters to Optimize:**
1. **Revenue Sharing**: α ∈ [0.1, 0.9]
2. **Flexibility Factor**: β ∈ [0.8, 1.2]
3. **Safety Stock**: γ ∈ [1.0, 2.5]
4. **Information Sharing**: δ ∈ [0, 100]
5. **Lead Time Buffer**: λ ∈ [0, 1]

### Implementation Steps

1. **Model Setup**: Define player strategies and utility functions
2. **Game Analysis**: Find Nash equilibrium using best response functions
3. **Contract Optimization**: Optimize contract parameters for bullwhip reduction
4. **Validation**: Test equilibrium stability and robustness
5. **Comparison**: Compare with traditional (non-coordinated) approach

### Python Implementation

In [1]:
from typing import Tuple, List, Dict, Set
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import List, Tuple, Dict
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
@dataclass
class SupplyChainPlayer:
    """Represents a player in the supply chain game"""
    name: str
    cost_per_unit: float
    holding_cost_rate: float
    stockout_cost: float
    lead_time: int
    
    def calculate_utility(self, demand: np.ndarray, orders: np.ndarray, 
                          contract_params: Dict) -> float:
        """Calculate player utility based on demand and orders"""
        # Calculate revenue
        revenue = np.sum(demand) * contract_params.get('price', 100)
        
        # Calculate costs
        production_cost = np.sum(orders) * self.cost_per_unit
        
        # Calculate inventory costs (simplified)
        avg_inventory = np.mean(orders) * 0.5  # Simplified average inventory
        holding_cost = avg_inventory * self.holding_cost_rate
        
        # Calculate bullwhip cost
        bullwhip_ratio = np.std(orders) / (np.std(demand) + 1e-6)
        bullwhip_cost = bullwhip_ratio * self.stockout_cost * 0.1
        
        # Apply contract parameters
        revenue_sharing = contract_params.get('revenue_sharing', 0.5)
        info_bonus = contract_params.get('info_bonus', 0)
        
        utility = (revenue * revenue_sharing + info_bonus - 
                  production_cost - holding_cost - bullwhip_cost)
        
        return utility

In [ ]:
class GameTheoreticContractDesigner:
    """Designs contracts using game-theoretic approach"""
    
    def __init__(self, players: List[SupplyChainPlayer]):
        self.players = players
        self.demand_history = []
        self.contract_history = []
        
    def generate_demand(self, periods: int, base_demand: int = 1000, 
                       noise_std: float = 200) -> np.ndarray:
        """Generate stochastic demand pattern"""
        np.random.seed(42)  # For reproducibility
        
        # Base demand with random walk and noise
        demand = np.zeros(periods)
        demand[0] = base_demand
        
        for t in range(1, periods):
            # Random walk component
            trend = np.random.normal(0, 50)
            # Seasonal component
            seasonal = 100 * np.sin(2 * np.pi * t / 12)
            # Random noise
            noise = np.random.normal(0, noise_std)
            
            demand[t] = max(100, demand[t-1] + trend + seasonal + noise)
        
        self.demand_history = demand
        return demand
    
    def calculate_orders(self, demand: np.ndarray, 
                        contract_params: Dict) -> Dict[str, np.ndarray]:
        """Calculate orders for each player based on contract"""
        orders = {}
        
        # Simplified order calculation based on demand and contract
        flexibility = contract_params.get('flexibility', 1.0)
        safety_stock = contract_params.get('safety_stock', 1.5)
        
        for player in self.players:
            # Order quantity with safety stock and flexibility
            base_order = demand * safety_stock
            order_variation = np.random.normal(0, 0.1 * demand)  # Order variation
            
            orders[player.name] = base_order * flexibility + order_variation
            orders[player.name] = np.maximum(0, orders[player.name])  # Non-negative
        
        return orders
    
    def find_nash_equilibrium(self, demand: np.ndarray, 
                             param_ranges: Dict) -> Dict:
        """Find Nash equilibrium for contract parameters"""
        best_params = {}
        best_utility = -np.inf
        
        # Grid search for equilibrium (simplified)
        revenue_sharing_range = param_ranges.get('revenue_sharing', [0.3, 0.5, 0.7])
        flexibility_range = param_ranges.get('flexibility', [0.8, 1.0, 1.2])
        safety_stock_range = param_ranges.get('safety_stock', [1.0, 1.5, 2.0])
        
        for rs in revenue_sharing_range:
            for flex in flexibility_range:
                for ss in safety_stock_range:
                    # Test contract parameters
                    contract_params = {
                        'revenue_sharing': rs,
                        'flexibility': flex,
                        'safety_stock': ss,
                        'info_bonus': 50,
                        'price': 100
                    }
                    
                    # Calculate orders and utilities
                    orders = self.calculate_orders(demand, contract_params)
                    
                    # Check if this is an equilibrium (all players prefer it)
                    total_utility = 0
                    is_equilibrium = True
                    
                    for player in self.players:
                        utility = player.calculate_utility(demand, 
                                                          orders[player.name], 
                                                          contract_params)
                        total_utility += utility
                        
                        # Check if player would deviate (simplified check)
                        if utility < 0:  # Player would not participate
                            is_equilibrium = False
                            break
                    
                    if is_equilibrium and total_utility > best_utility:
                        best_utility = total_utility
                        best_params = contract_params.copy()
                        best_params['total_utility'] = total_utility
        
        return best_params
    
    def calculate_bullwhip_effect(self, demand: np.ndarray, 
                                 orders: Dict[str, np.ndarray]) -> float:
        """Calculate overall bullwhip effect"""
        demand_std = np.std(demand)
        
        if demand_std == 0:
            return 1.0
        
        # Calculate average order variability across all players
        order_stds = []
        for player_orders in orders.values():
            order_stds.append(np.std(player_orders))
        
        avg_order_std = np.mean(order_stds)
        bullwhip_ratio = avg_order_std / demand_std
        
        return bullwhip_ratio

In [ ]:
# Initialize the supply chain
players = [
    SupplyChainPlayer("Supplier", cost_per_unit=30, holding_cost_rate=0.15, 
                      stockout_cost=500, lead_time=3),
    SupplyChainPlayer("Manufacturer", cost_per_unit=50, holding_cost_rate=0.20, 
                      stockout_cost=800, lead_time=2),
    SupplyChainPlayer("Distributor", cost_per_unit=70, holding_cost_rate=0.25, 
                      stockout_cost=1200, lead_time=1),
    SupplyChainPlayer("Retailer", cost_per_unit=85, holding_cost_rate=0.30, 
                      stockout_cost=1500, lead_time=0)
]

# Create contract designer
designer = GameTheoreticContractDesigner(players)

# Generate demand pattern
periods = 52  # One year of weekly data
demand = designer.generate_demand(periods, base_demand=1000, noise_std=200)

print(f"Generated {periods} periods of demand")
print(f"Demand statistics: Mean={np.mean(demand):.1f}, Std={np.std(demand):.1f}")
print(f"Demand range: {np.min(demand):.1f} to {np.max(demand):.1f}")

Generated 52 periods of demand
Demand statistics: Mean=777.8, Std=466.1
Demand range: 100.0 to 1966.3


In [ ]:
# Define parameter ranges for optimization
param_ranges = {
    'revenue_sharing': [0.3, 0.4, 0.5, 0.6, 0.7],
    'flexibility': [0.8, 0.9, 1.0, 1.1, 1.2],
    'safety_stock': [1.0, 1.25, 1.5, 1.75, 2.0]
}

# Find Nash equilibrium
equilibrium_params = designer.find_nash_equilibrium(demand, param_ranges)

print("=== Nash Equilibrium Contract Parameters ===")
for key, value in equilibrium_params.items():
    if key != 'total_utility':
        print(f"{key}: {value}")
print(f"Total Supply Chain Utility: ${equilibrium_params.get('total_utility', 0):,.2f}")

=== Nash Equilibrium Contract Parameters ===
revenue_sharing: 0.7
flexibility: 0.8
safety_stock: 1.0
info_bonus: 50
price: 100
Total Supply Chain Utility: $3,695,022.46


In [ ]:
# Calculate orders and bullwhip effect with equilibrium parameters
orders = designer.calculate_orders(demand, equilibrium_params)
bullwhip_effect = designer.calculate_bullwhip_effect(demand, orders)

print(f"\n=== Bullwhip Effect Analysis ===")
print(f"Demand Standard Deviation: {np.std(demand):.2f}")
print(f"Average Order Standard Deviation: {np.mean([np.std(orders[p.name]) for p in players]):.2f}")
print(f"Bullwhip Ratio: {bullwhip_effect:.3f}")

# Compare with traditional approach (no coordination)
traditional_params = {
    'revenue_sharing': 0.5,
    'flexibility': 1.0,
    'safety_stock': 1.5,
    'info_bonus': 0,
    'price': 100
}

traditional_orders = designer.calculate_orders(demand, traditional_params)
traditional_bullwhip = designer.calculate_bullwhip_effect(demand, traditional_orders)

print(f"\nTraditional Approach Bullwhip Ratio: {traditional_bullwhip:.3f}")
print(f"Bullwhip Reduction: {((traditional_bullwhip - bullwhip_effect) / traditional_bullwhip * 100):.1f}%")


=== Bullwhip Effect Analysis ===
Demand Standard Deviation: 466.05
Average Order Standard Deviation: 387.38
Bullwhip Ratio: 0.831

Traditional Approach Bullwhip Ratio: 1.491
Bullwhip Reduction: 44.3%


In [ ]:
# Create comprehensive visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Game-Theoretic Contract Design Analysis', fontsize=16, fontweight='bold')

# Plot 1: Demand Pattern
axes[0, 0].plot(demand, 'b-', linewidth=2, label='Customer Demand')
axes[0, 0].set_title('Demand Pattern Over Time', fontweight='bold')
axes[0, 0].set_xlabel('Period')
axes[0, 0].set_ylabel('Demand Units')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].legend()

# Plot 2: Order Variability Comparison
periods_plot = range(min(20, len(demand)))  # Show first 20 periods for clarity
for i, player in enumerate(players):
    axes[0, 1].plot(periods_plot, orders[player.name][:20], 
                    linewidth=2, label=player.name, alpha=0.8)
axes[0, 1].plot(periods_plot, demand[:20], 'k--', linewidth=2, label='Demand')
axes[0, 1].set_title('Order Patterns vs Customer Demand', fontweight='bold')
axes[0, 1].set_xlabel('Period')
axes[0, 1].set_ylabel('Units')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].legend(bbox_to_anchor=(1.05, 1), loc='upper left')

# Plot 3: Bullwhip Effect Comparison
methods = ['Traditional', 'Game-Theoretic']
bullwhip_ratios = [traditional_bullwhip, bullwhip_effect]
colors = ['lightcoral', 'lightgreen']

bars = axes[1, 0].bar(methods, bullwhip_ratios, color=colors, alpha=0.7)
axes[1, 0].set_title('Bullwhip Effect Comparison', fontweight='bold')
axes[1, 0].set_ylabel('Bullwhip Ratio')
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar, ratio in zip(bars, bullwhip_ratios):
    height = bar.get_height()
    axes[1, 0].text(bar.get_x() + bar.get_width()/2., height + 0.01,
                    f'{ratio:.3f}', ha='center', va='bottom', fontweight='bold')

# Plot 4: Contract Parameter Impact
param_names = ['Revenue\nSharing', 'Flexibility', 'Safety\nStock']
traditional_values = [0.5, 1.0, 1.5]
equilibrium_values = [equilibrium_params['revenue_sharing'], 
                      equilibrium_params['flexibility'], 
                      equilibrium_params['safety_stock']]

x = np.arange(len(param_names))
width = 0.35

bars1 = axes[1, 1].bar(x - width/2, traditional_values, width, 
                        label='Traditional', color='lightcoral', alpha=0.7)
bars2 = axes[1, 1].bar(x + width/2, equilibrium_values, width, 
                        label='Game-Theoretic', color='lightgreen', alpha=0.7)

axes[1, 1].set_title('Contract Parameter Comparison', fontweight='bold')
axes[1, 1].set_ylabel('Parameter Value')
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(param_names)
axes[1, 1].grid(True, alpha=0.3, axis='y')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Sensitivity Analysis
print("=== Sensitivity Analysis ===")

# Test different demand variability levels
noise_levels = [100, 200, 300, 400]
bullwhip_results = []

for noise in noise_levels:
    test_demand = designer.generate_demand(periods, base_demand=1000, noise_std=noise)
    test_orders = designer.calculate_orders(test_demand, equilibrium_params)
    test_bullwhip = designer.calculate_bullwhip_effect(test_demand, test_orders)
    bullwhip_results.append(test_bullwhip)
    print(f"Noise Std={noise}: Bullwhip Ratio = {test_bullwhip:.3f}")

# Plot sensitivity results
plt.figure(figsize=(10, 6))
plt.plot(noise_levels, bullwhip_results, 'go-', linewidth=2, markersize=8)
plt.xlabel('Demand Noise Standard Deviation')
plt.ylabel('Bullwhip Ratio')
plt.title('Bullwhip Effect Sensitivity to Demand Variability', fontweight='bold')
plt.grid(True, alpha=0.3)
plt.show()

=== Sensitivity Analysis ===
Noise Std=100: Bullwhip Ratio = 0.841
Noise Std=200: Bullwhip Ratio = 0.835
Noise Std=300: Bullwhip Ratio = 0.831
Noise Std=400: Bullwhip Ratio = 0.828


## Results Summary

### Key Findings

1. **Equilibrium Contract Parameters**:
   - Revenue sharing ratio optimized for fair profit distribution
   - Flexibility factor balanced to reduce order amplification
   - Safety stock multiplier minimized while maintaining service levels

2. **Bullwhip Effect Reduction**:
   - Significant reduction in demand variability amplification
   - Improved supply chain stability and predictability
   - Better coordination between supply chain partners

3. **Economic Benefits**:
   - Higher total supply chain utility
   - Reduced inventory holding costs
   - Improved service levels and customer satisfaction

### When to Use This Tier

**Use Tier 1 when:**
- Supply chain has **few tiers** (2-4 participants)
- **Cost parameters are well-known** and relatively stable
- **Theoretical understanding** of contract mechanisms is needed
- **Benchmark performance** is required for evaluation
- **Academic research** or teaching game theory applications

**Consider higher tiers when:**
- Supply chain has **many tiers** with complex interactions
- **Real-time adaptation** is required for dynamic environments
- **Advanced algorithms** are needed for large-scale optimization
- **Machine learning** can improve contract design
- **Digital twin** capabilities are available for simulation

### Limitations and Next Steps

1. **Computational Complexity**: Grid search approach may not scale to large parameter spaces
2. **Simplified Assumptions**: Real-world supply chains have more complex dynamics
3. **Static Analysis**: Does not account for dynamic learning and adaptation

**Next Tier Preview**: Tier 2 introduces iterative contract parameter optimization with advanced algorithms for improved performance and scalability.